# Fit and evaluate a pump-power model — reference

Compare simple regression and a physical feature on independent synthetic operating designs. Use validation for selection and reserve a locked test.

All examples are explicitly synthetic teaching examples, not validated machinery data.
Run from a clean kernel, top to bottom. Python 3.10+ and NumPy 1.26+ are the declared target environment. See the course validation file for actual tested versions. No internet or AI API is used by these notebooks after installing dependencies.
Use the starter before consulting the reference. Record permitted AI assistance and explain your own choices.


In [ ]:
import sys
import numpy as np
print('Python', sys.version.split()[0], 'NumPy', np.__version__)
rng=np.random.default_rng(42)
n=150
flow=rng.uniform(0.006,0.035,n)
dp=rng.uniform(80000,200000,n)
X=np.column_stack([flow,dp])
y=flow*dp/0.72/1000 + rng.normal(0,0.12,n)
order=rng.permutation(n)
tr,va,te=order[:90],order[90:120],order[120:]
print('Independent synthetic designs. train/validation/test:',len(tr),len(va),len(te))
print('X units: m^3/s and Pa. y unit: kW.')


## 1. Fit a repeatable transform and model

Define fit_linear(A,b), returning training mean, safe training standard deviation and coefficients. A shape (n,d), b shape (n,). Build an intercept column and use np.linalg.lstsq. Define predict(model,A) returning (n,).

In [ ]:
def fit_linear(A,b):
    mean=A.mean(axis=0); sd=A.std(axis=0)
    sd=np.where(sd>0,sd,1.0)
    Z=np.column_stack([np.ones(len(A)),(A-mean)/sd])
    coef=np.linalg.lstsq(Z,b,rcond=None)[0]
    return mean,sd,coef
def predict(model,A):
    mean,sd,coef=model
    return np.column_stack([np.ones(len(A)),(A-mean)/sd])@coef


## 2. Compare on validation

Fit two candidates: raw X and physical feature H=flow*dp/1000 with shape (150,1). Compare validation MAE with a constant training-mean baseline. Select without using te.

In [ ]:
def choose_candidate(X,y,tr,va):
    features={'raw':X,'hydraulic':(X[:,0]*X[:,1]/1000).reshape(-1,1)}
    results={}
    for name,A in features.items():
        m=fit_linear(A[tr],y[tr]); mae=np.mean(np.abs(y[va]-predict(m,A[va])))
        results[name]=(mae,m,A)
        print(name,'validation MAE kW:',round(mae,4))
    print('training-mean baseline validation MAE kW:',round(np.mean(np.abs(y[va]-y[tr].mean())),4))
    chosen=min(results,key=lambda k:results[k][0])
    _,model,A=results[chosen]
    print('Selected using validation:',chosen)
    return chosen,model,A
chosen,model,A=choose_candidate(X,y,tr,va)


## 3. Locked test and metrics

Refit the selected specification on tr+va, then evaluate te once. Return MAE, RMSE and R². Handle SST=0 explicitly. Baselines use tr+va only. Residual means observation minus prediction.

In [ ]:
def metrics(y,p):
    r=y-p; sst=np.sum((y-y.mean())**2)
    return dict(MAE=float(np.abs(r).mean()), RMSE=float(np.sqrt((r*r).mean())),
                R2=float(1-np.sum(r*r)/sst) if sst>0 else None)
dev=np.concatenate([tr,va]); locked=fit_linear(A[dev],y[dev]); p=predict(locked,A[te])
print('Locked model test metrics:',metrics(y[te],p))
print('Training-mean baseline:',metrics(y[te],np.full(len(te),y[dev].mean())))
physics=X[te,0]*X[te,1]/0.72/1000
print('Known synthetic physics baseline:',metrics(y[te],physics))
assert np.isclose(metrics(np.array([1.,-2.,3.]),np.zeros(3))['MAE'],2.0)
assert metrics(np.ones(3),np.ones(3))['R2'] is None
print('Independent physical check: 100000 Pa * 0.02 m^3/s / 0.8 =',100000*0.02/0.8,'W')


## Report
Describe independent synthetic designs, split sizes, chosen features, validation choice, test metrics and baseline. Explain why this random split is not the default for forecasting or transfer across machines. State limitations; do not infer validated pump performance.
